# T2-E2: bounded nonlinear observer study

Upload **T2_E2_Nonlinear_Observer.zip**. This notebook verifies the archived study, reproduces the fixed development sensitivity check, and permits the main empirical evaluation only when that check passes. No GPU or new model training is required; observer networks are fitted on CPU.

The protocol, six development-exposed Transformer checkpoints, exact examples, observer weights, predictions, metrics, and verification records are included. Reserved model seeds remain sealed. A deterministic rerun is a reproduction, not an independent replication.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, subprocess, sys, tempfile, zipfile
uploaded=files.upload()
assert set(uploaded)=={'T2_E2_Nonlinear_Observer.zip'}, 'Upload exactly the supplied ZIP.'
work=Path(tempfile.mkdtemp(prefix='t2_e2_'))
with zipfile.ZipFile('T2_E2_Nonlinear_Observer.zip') as archive:
    for member in archive.infolist():
        name=PurePosixPath(member.filename)
        assert not name.is_absolute() and '..' not in name.parts
        assert (member.external_attr >> 16) & 0o170000 != 0o120000
        assert (work/member.filename).resolve().is_relative_to(work.resolve())
    archive.extractall(work)
root=work/'T2_E2_Nonlinear_Observer'
manifest=json.loads((root/'PACKAGE_SHA256.json').read_text())
for name,digest in manifest.items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest()==digest,name
print(f'Archive verified: {len(manifest)} files')


## Verify the original results

All experiment commands use fresh Python subprocesses with the pinned packages. The notebook itself does not execute a private uploaded pickle or require external Drive access.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'requirements.txt')],check=True)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=root,check=True)
run('verify_methods.py')
run('verify_results.py','--folder','development','--development')
if (root/'results/study_result.json').exists():
    run('verify_results.py','--folder','results')
archived_go=json.loads((root/'development/development_result.json').read_text())['go_to_main']


## Reproduce the development sensitivity check

This uses fresh copies of the same deterministic development data: real model representations with an independent injected history coordinate and synthetic targets. The oracle relative MSE gains are 0%, 5%, and 20%. The null must fail and the 20% case must pass the frozen complete screen. There is no tuning or repetition if the check fails.


In [ ]:
run('-u','run_experiment.py','development','--development','reproduced_development')
run('verify_results.py','--folder','reproduced_development','--development')
reproduced_go=json.loads((root/'reproduced_development/development_result.json').read_text())['go_to_main']
assert reproduced_go==archived_go,'The development decision differs; inspect environment and numeric differences.'
print('Development gate:',reproduced_go)


## Freeze, fit, and evaluate the empirical comparison

When the development gate passes, the scientific protocol remains unchanged. The notebook preserves the original main freeze under another filename in this disposable extraction, creates the reproduction's freeze, fits all probes using training/validation only, and then evaluates held-out outcomes once. The output folders are new; archived metrics and weights remain available.


In [ ]:
if reproduced_go:
    freeze=root/'MAIN_FREEZE.json'
    if freeze.exists():freeze.rename(root/'ARCHIVED_MAIN_FREEZE.json')
    run('run_experiment.py','freeze','--development','reproduced_development','--out','reproduced_results')
    run('-u','run_experiment.py','fit','--development','reproduced_development','--out','reproduced_results')
    run('-u','run_experiment.py','evaluate','--development','reproduced_development','--out','reproduced_results')
    run('verify_results.py','--folder','reproduced_results')
else:
    print('The prespecified stopping rule ends the study before any empirical test opens.')


In [ ]:
comparison=r"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
for old,new in [('development','reproduced_development'),('results','reproduced_results')]:
    if not Path(new).exists():continue
    a=pd.read_csv(Path(old)/'probe_metrics.csv')
    b=pd.read_csv(Path(new)/'probe_metrics.csv')
    assert list(a.columns)==list(b.columns)
    assert a[['seed','axis','arm','eligible']].equals(b[['seed','axis','arm','eligible']])
    np.testing.assert_allclose(a[['mse','mae','r2']],b[['mse','mae','r2']],rtol=1e-3,atol=1e-4,equal_nan=True)
    da=json.loads((Path(old)/'decisions.json').read_text())
    db=json.loads((Path(new)/'decisions.json').read_text())
    assert da==db,'A frozen decision differs from the archived run.'
    print(old,': scores match within numeric tolerance; all decisions match.')
    print(json.dumps(db,indent=2))
"""
run('-c',comparison)


## Inspect the report and download the reproduction

The main comparison concerns remaining true-label NLL reduction with these particular trained predictors and six fixed models. Conditional bootstrap intervals do not establish generalization to new models. Sensitivity to an injected linear signal is not guaranteed sensitivity to natural history effects.


In [ ]:
from IPython.display import Image,Markdown,display
import shutil
display(Markdown((root/'T2_E2_Report.md').read_text()))
figure=root/'T2_E2_History_Effects.png'
if figure.exists():display(Image(filename=str(figure)))
if reproduced_go:
    filename=shutil.make_archive(str(work/'T2_E2_reproduced_results'),'zip',root/'reproduced_results')
    files.download(filename)
